# Modeling
Entraînement et comparaison des modèles Random Forest et XGBoost sur le dataset final préparé.

## Imports et chargement

In [ ]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

df = pd.read_csv("../data/processed/dataset_final.csv")
df["DateHeure"] = pd.to_datetime(df["DateHeure"])
df = df.sort_values("DateHeure").reset_index(drop=True)

FEATURES = [
    "Temperature",
    "Humidite",
    "Pression",
    "VitesseVent",
    "Heure_sin",
    "Heure_cos",
    "Mois_sin",
    "Mois_cos",
    "IsWeekend",
    "wind_dir_sin",
    "wind_dir_cos",
    "cloud_cover",
    "precipitation_bin",
    "AQI_mean_6h",
] + [c for c in df.columns if c.startswith("ville_")]

TARGET = "AqiGlobal"

X = df[FEATURES]
y = df[TARGET]

tscv = TimeSeriesSplit(n_splits=3)

print(f"Dataset chargé : {len(df)} lignes. {len(FEATURES)} features")
print(f"Variable cible : {TARGET}")

Dataset chargé : 10526 lignes. 25 features
Variable cible : AqiGlobal


## 1. Modèle 1 - Random Forest

**Justification des paramètres :**

| Paramètre | Valeur | Justification |
|-----------|--------|---------------|
| `n_estimators` | 200 | Suffisant pour 10K lignes. au-delà le gain est marginal |
| `max_depth` | 12 | Évite l'overfitting sur 3 mois de données |
| `min_samples_split` | 10 | Empêche les splits sur trop peu de données |
| `min_samples_leaf` | 5 | Robustesse aux trous temporels |
| `max_features` | 0.5 | Réduit la variance en ne regardant que 50% des features à chaque split |
| `bootstrap` | True | Meilleure généralisation via le bagging |

In [13]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)

rf_scores = cross_val_score(rf_model, X, y, cv=tscv, scoring="r2")

print(f"Random Forest R² par split : {rf_scores.round(4)}")
print(f"Random Forest R² moyen     : {rf_scores.mean():.4f}")
print(f"Random Forest R² std       : {rf_scores.std():.4f}")

Random Forest R² par split : [0.8184 0.8577 0.8724]
Random Forest R² moyen     : 0.8495
Random Forest R² std       : 0.0228


### Observations - Random Forest
R² moyen : **0.8487** est largement au‑dessus du seuil exigé (0.5 par le jury). Le modèle explique ~85% de la variance de l'AQI.

Progression des splits :
- Split 1 (train = mars‑avril) : **0.8185**
- Split 2 (train = mars‑mai)   : **0.8564**
- Split 3 (train = mars‑juin)  : **0.8711**

Interprétation : le R² augmente avec la taille du train, ce qui indique une amélioration lorsque le modèle voit plus de données et suggère l'absence d'overfitting.

Écart‑type : **0.0221** très faible, les scores sont cohérents d'un split à l'autre; le modèle est stable.

## 2. Modèle 2 - XGBoost

**Justification des paramètres :**

| Paramètre | Valeur | Justification |
|-----------|--------|---------------|
| `n_estimators` | 200 | Nombre de rounds de boosting |
| `max_depth` | 6 | Arbres moins profonds que RF car XGBoost booste séquentiellement |
| `learning_rate` | 0.1 | Plus robuste qu'à 0.3. généralise mieux |
| `subsample` | 0.8 | Réduit l'overfitting sur notre petit historique |
| `colsample_bytree` | 0.8 | Réduit l'overfitting en échantillonnant les features |
| `gamma` | 0.2 | Régularisation supplémentaire. évite les splits inutiles |
| `reg_alpha` | 0.1 | Régularisation L1 légère |
| `reg_lambda` | 1 | Régularisation L2 standard |

In [14]:
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.2,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1,
)

xgb_scores = cross_val_score(xgb_model, X, y, cv=tscv, scoring="r2")

print(f"XGBoost R² par split : {xgb_scores.round(4)}")
print(f"XGBoost R² moyen     : {xgb_scores.mean():.4f}")
print(f"XGBoost R² std       : {xgb_scores.std():.4f}")

XGBoost R² par split : [0.7562 0.8258 0.8504]
XGBoost R² moyen     : 0.8108
XGBoost R² std       : 0.0399


### Observations - XGBoost
- R carré moyen de 0.8111. au-dessus du seuil jury de 0.5 mais inférieur à Random Forest (0.8487). La progression entre les splits est visible (0.75 -> 0.83 -> 0.85) ce qui confirme que XGBoost bénéficie de plus de données. mais il démarre moins bien que Random Forest sur le premier split avec seulement 2633 lignes d'entraînement.
- La std de 0.0434 est deux fois plus élevée que celle de Random Forest (0.0221). ce qui traduit une moins grande stabilité entre les splits. XGBoost est plus sensible à la quantité de données disponibles. ce qui est cohérent avec son fonctionnement séquentiel par boosting.
- Les deux modèles sont valides pour le jury. mais Random Forest est retenu comme modèle final car il offre un meilleur R carré moyen et une plus grande stabilité sur notre historique de 3 mois.

## 3. Comparaison et choix du modèle final

In [15]:
print("COMPARAISON DES MODÈLES")
print(f"Random Forest R² moyen : {rf_scores.mean():.4f} (std={rf_scores.std():.4f})")
print(f"XGBoost       R² moyen : {xgb_scores.mean():.4f} (std={xgb_scores.std():.4f})")
print()

best_name = "Random Forest" if rf_scores.mean() >= xgb_scores.mean() else "XGBoost"
best_model = rf_model if rf_scores.mean() >= xgb_scores.mean() else xgb_model

print(f"Modèle retenu : {best_name}")
print()
print("Critères de sélection :")
print(
    f"  R² > 0.5 (exigence jury)  : {'OUI' if max(rf_scores.mean(), xgb_scores.mean()) > 0.5 else 'NON'}"
)
print(f"  Meilleur R² moyen: {best_name}")

COMPARAISON DES MODÈLES
Random Forest R² moyen : 0.8495 (std=0.0228)
XGBoost       R² moyen : 0.8108 (std=0.0399)

Modèle retenu : Random Forest

Critères de sélection :
  R² > 0.5 (exigence jury)  : OUI
  Meilleur R² moyen: Random Forest


### Observations - Comparaison

Random Forest est retenu avec un R² moyen de 0.8487 contre 0.8111 pour XGBoost. Les deux modèles dépassent largement l'exigence jury de R² > 0.5. ce qui valide notre approche de préparation des données.

Random Forest s'impose sur les deux critères décisifs : meilleur R² moyen (+3.7 points) et meilleure stabilité (std 0.0221 vs 0.0434). Sur un dataset de 3 mois avec des trous temporels. le bagging de Random Forest s'avère plus adapté que le boosting séquentiel de XGBoost qui nécessite davantage de données pour exprimer son potentiel.

Ce résultat est aussi une bonne nouvelle pour le jury : Random Forest est l'algorithme le plus explicable des deux grâce à son feature_importance. On pourra montrer visuellement quelles variables météo expliquent le mieux l'AQI. ce qui répond directement aux attentes métier de GoodAir.

## 4. Entraînement final sur tout le dataset

In [16]:
# Entraînement sur la totalité du dataset
best_model.fit(X, y)
print(f"{best_name} entraîné sur {len(X)} lignes.")

# Sauvegarde du meilleur modèle
joblib.dump(best_model, "../src/ml/models/aqi_model.pkl")
print("Modèle sauvegardé : src/ml/models/aqi_model.pkl")

# Sauvegarde individuelle
rf_model.fit(X, y)
joblib.dump(rf_model, "../src/ml/models/random_forest_aqi.pkl")
xgb_model.fit(X, y)
xgb_model.save_model("../src/ml/models/xgboost_aqi.json")
print("Modèles individuels sauvegardés.")

Random Forest entraîné sur 10526 lignes.
Modèle sauvegardé : src/ml/models/aqi_model.pkl
Modèles individuels sauvegardés.


## Pourquoi ces choix de modèles ?

### Pourquoi pas les modèles de séries temporelles (ARIMA, SARIMA, Prophet) ?

Les modèles de séries temporelles comme ARIMA ou Prophet sont conçus pour prédire
une valeur future uniquement à partir des valeurs passées de cette même variable.
Dans notre cas cela voudrait dire prédire l'indice de qualité de l'air uniquement
à partir des indices de qualité de l'air précédents. sans tenir compte de la météo.

Deux problèmes rendent ces modèles inadaptés à notre projet.

Le premier problème est nos trous temporels. ARIMA exige une série temporelle
continue et régulièrement espacée. Notre pipeline tourne sur un ordinateur personnel
qui a été en veille certaines nuits et certains week-ends. Ces discontinuités
rendent ARIMA inutilisable sans imputation artificielle des données manquantes.
Nous avons délibérément refusé de fabriquer des données pour combler ces trous
afin de préserver l'intégrité scientifique du projet.

Le deuxième problème est que notre prédiction est multivariée. Nous voulons
prédire l'indice de qualité de l'air en utilisant des variables météo simultanées
comme la température. le vent et les précipitations. Ces modèles ne peuvent pas
exploiter naturellement ces variables complémentaires.

### Pourquoi pas la régression linéaire ?

La régression linéaire suppose qu'il existe une relation directe et proportionnelle
entre chaque variable et l'indice de qualité de l'air. Par exemple elle supposerait
que chaque degré de température supplémentaire fait toujours monter l'indice du
même montant. quelle que soit l'heure. la ville ou la saison.

Nos analyses exploratoires ont montré que ce n'est pas le cas. Les relations entre
la météo et la qualité de l'air sont non linéaires et dépendent du contexte.
Un vent fort fait baisser l'indice en journée mais peut ne pas avoir le même effet
la nuit. La régression linéaire ne peut pas capturer ces interactions complexes.

De plus notre variable cible. l'indice de qualité de l'air. a une distribution
fortement asymétrique avec des pics extrêmes. La régression linéaire est très
sensible à ces valeurs extrêmes et aurait produit des prédictions instables.

### Pourquoi Random Forest plutôt que XGBoost ?

Mathématiquement et sur des datasets volumineux. XGBoost est généralement
supérieur à Random Forest. C'est une réalité reconnue dans la communauté
data science. Cependant sur notre projet les résultats ont été les suivants.

| Modèle | R² moyen | Stabilité (écart-type) |
|--------|----------|------------------------|
| Random Forest | 0.8487 | 0.0221 |
| XGBoost | 0.8111 | 0.0434 |

Random Forest obtient un meilleur R² et une meilleure stabilité sur nos données.

L'explication est simple. XGBoost apprend de manière séquentielle en corrigeant
progressivement ses erreurs. Cette approche nécessite beaucoup de données pour
exprimer tout son potentiel. Sur notre premier split qui contient seulement
2633 lignes d'entraînement. XGBoost obtient un R² de 0.75 contre 0.82 pour
Random Forest. Il manque de données pour bien apprendre.

Random Forest quant à lui construit plusieurs centaines d'arbres de décision
indépendants et fait la moyenne de leurs prédictions. Cette approche par
consensus est naturellement plus robuste sur des petits datasets avec des
trous temporels. exactement notre situation.

En résumé. sur un historique de 3 mois avec des discontinuités. Random Forest
est plus adapté que XGBoost. Si nous avions 2 ans de données continues.
XGBoost aurait probablement été plus performant.